# CERNAL quickstart

A trigger sequence goes in; ranked RNA logic circuit designs, best first, come out.
This notebook walks through the same five calls the Python client wraps —
`design · status · results · artifact · capabilities` — end to end, then a
constrained + custom-scored run, a cost-first parameter sweep, and a couple of
plots of the results.

**Fill in `CERNAL_API_KEY` and `base_url` below before running.** Mint a key from
the *API Keys* page in the app.

> If `GET /api/version` on your deployment reports `"engine": "MockEngine"`, every
> number below is deterministic simulated science, not a real folding prediction —
> useful for learning the API, not for a lab notebook.

In [ ]:
%pip install -q cernal[pandas] matplotlib

## 1 · Authenticate

Holds a key and a base URL. Nothing else is stateful.

In [ ]:
import os
from cernal import Client

os.environ.setdefault("CERNAL_API_KEY", "cern_live_replace_me")

c = Client(api_key=os.environ["CERNAL_API_KEY"], base_url="https://your-cernal-host")
c.capabilities()["engine"]  # confirms the key works and shows Mock vs real

## 2 · A ranked design, in one call

`Job.wait()` polls with backoff (2s, growing to 15s) until the run finishes.
`.to_dataframe()` needs pandas; `.to_dicts()` always works with no extra dependency.

In [ ]:
job = c.design(trigger_sequence="AUGGCUAAGCUUAACGGAUCC", organism="ecoli")
df = job.wait().to_dataframe()
df.head()

## 3 · Constrained, with custom scoring

`scoring.weights` re-weights the nine metrics. `weight: 0.0` means *measure it,
report it, just don't rank on it* — the raw value stays visible in the
decomposition. `hard_filters` adds a disqualifying threshold on top of the
profile's own two (`predicted_leakage <= 0.85`, `state_separation >= 0.5`).

In [ ]:
job = c.design(
    trigger_sequence="AUGGCUAAGCUUAACGGAUCCAUGGCUAAGCUUAAC",
    organism="ecoli",
    gate_families=["toehold"],
    constraints={
        "max_triggers": 2,
        "min_separation": 1.0,
        "max_p_adj": 0.01,
        "trigger_lengths": [30, 36],
        "standard": "RFC10",
    },
    scoring={
        "weights": {"predicted_leakage": 4.0, "gc_content": 0.0},
        "hard_filters": [{"metric": "dynamic_range", "minimum": 10.0}],
    },
    budget={"max_designs": 50_000, "max_runtime_seconds": 1800},
    seed=42,
    top_n=25,
)
print(job.estimate)                 # from the 202, before waiting
results = job.wait()
print(results.resolved["scoring_profile"])   # "custom-<8-char hash>" — reproducible
best = results.best()
best

## 4 · Cost a sweep before running any of it

`dry_run=True` estimates without submitting — no queue slot, no compute spent.
Sweep every organism you care about first, then flip `dry_run` off for the one
run you actually want.

In [ ]:
cheap = Client(api_key=os.environ["CERNAL_API_KEY"], base_url="https://your-cernal-host", dry_run=True)

estimates = {}
for organism in ("ecoli", "yeast"):
    est = cheap.design(trigger_sequence="AUGGCUAAGCUUAACGGAUCC", organism=organism).estimate
    estimates[organism] = est
    print(f"{organism}: {est['designs']} designs, ~{est['seconds']}s ({est['confidence']})")

## 5 · Plot the score decomposition

`to_dataframe()` puts every metric's `raw_value` in its own column, so the
result drops straight into `matplotlib`/`pandas.plot` — no reshaping needed.

In [ ]:
import matplotlib.pyplot as plt

top5 = df.sort_values("overall_score", ascending=False).head(5)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(top5["id"].astype(str), top5["overall_score"], color="#c02b2b")
ax.set_xlabel("overall_score")
ax.set_title("Top 5 candidates by overall score")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

## 6 · Download an artifact

Fetched fresh — a prior `include_artifacts` at submission time isn't required.

In [ ]:
job.artifact("fasta").save("best.fa")
print(open("best.fa").read()[:200])

## Where to go next

- The full parameter reference and the 9 scoring metrics: `/api-docs` in the app.
- Every failure raises a typed exception — `AuthError`, `ValidationError`
  (carries `.did_you_mean`), `RateLimited` (carries `.retry_after`), `RunFailed`
  (carries `.error_summary`).
- Anything beyond these five calls — projects, datasets, annotations — is the full
  REST API: `/api/docs` (interactive) and `/api/openapi.json` (machine-readable).